In [37]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [38]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    for item in result.get('data', []):
        if 'summary' in item:
            normalized_data.append(item)

observed_src = pd.json_normalize(normalized_data)
observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
observed_src.drop_duplicates(subset='indicator', inplace=True)
observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]


Querying owner: HTOC Org


In [39]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,ip,legacyLink,tags.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,6755399460007699,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-11T15:44:17Z,4.0,68.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,103.147.246.222,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...",NaN,NaN,NaN,NaN,NaN,NaN,103.147.246.222
1,6755399442005037,2025-02-12T20:18:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-11T12:51:39Z,4.0,64.0,Suspicious attempts being made to authenticate...,...,47.237.115.193,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,47.237.115.193
2,6755399460003159,2025-06-30T17:15:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-11T12:50:59Z,3.0,74.0,RITM0594044,...,13.221.86.176,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,13.221.86.176
3,5629499560349693,2025-07-22T17:27:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-11T12:12:56Z,5.0,88.0,"On July 18, security researchers and CISA disc...",...,103.172.81.132,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1471424, 'name': 'UNC6337/STORM-2603',...",NaN,NaN,NaN,NaN,NaN,NaN,103.172.81.132
4,6755399460007958,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-11T11:31:07Z,4.0,68.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,103.125.173.75,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...",NaN,NaN,NaN,NaN,NaN,NaN,103.125.173.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1199,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,ACD R&F processed a malspam campaign with a Ne...,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...",NaN,NaN,NaN,https://www.shorturl.at/,NaN,www.shorturl.at/,www.shorturl.at/
1200,4303591,2023-03-03T13:53:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-24T23:25:10Z,3.0,72.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...",NaN,NaN,NaN,NaN,NaN,aka.ms/o0ukef,aka.ms/o0ukef
1201,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com,selligenttier.naylorcampaigns.com
1203,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,https://google,NaN,google,google


In [40]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250811.csv']
Loaded data from 1 files.


In [41]:
import warnings

# Extract API-tagged indicators
all_filtered = []
cutoff = pd.Timestamp.utcnow()

for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance(tags_data, list):
        tags_df = pd.json_normalize(tags_data)
        tags_df['name'] = tags_df['name'].str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
        api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)]
        if not api_tags.empty:
            all_tags_list = tags_df['name'].astype(str).tolist()
            api_tags['name'] = api_tags['name'].astype(str)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink','rating', 'confidence', 'id']:
                api_tags[col] = row.get(col)
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            all_filtered.append(api_tags)
            warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
filtered_tags = pd.concat(all_filtered, ignore_index=True)
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Prepare for matching
filtered_tags['OpDiv'] = filtered_tags['name'].str.replace(' Splunk API', '', regex=False)
obs_subset = observed_data_df[['indicator', 'OpDiv', 'obs_date']].drop_duplicates()
recent_tags = pd.merge(filtered_tags, obs_subset, left_on=['summary', 'OpDiv'], right_on=['indicator', 'OpDiv'], how='inner')

# Filter to recent
cutoff_naive = cutoff.tz_convert(None)
recent_tags = recent_tags[
    (recent_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (pd.to_datetime(recent_tags['obs_date'], errors='coerce') >= cutoff_naive - timedelta(days=1))
].copy()

# Partner processing
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
recent_tags.drop_duplicates(subset='summary', inplace=True)
recent_tags


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,tactics.data,tactics.count,platforms.data,platforms.count,OpDiv,indicator,obs_date,partner,partner_count,partners
0,6755399442005037,DHA Splunk API,2025-08-11T15:44:17Z,47.237.115.193,8695787,Suspicious attempts being made to authenticate...,Address,2025-02-12 20:18:21+00:00,2025-08-11T12:51:39Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,47.237.115.193,2025-08-11,DHA,7,"CMS, DHA, FDA, HHS, IHS, OS, VA"
7,5629499560349693,VA Splunk API,2025-08-10T23:06:33Z,103.172.81.132,224,"On July 18, security researchers and CISA disc...",Address,2025-07-22 17:27:41+00:00,2025-08-11T12:12:56Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,VA,103.172.81.132,2025-08-11,VA,1,VA
8,6755399460007958,DHA Splunk API,2025-08-11T15:44:17Z,103.125.173.75,5,"Details\nIn late June 2025, Iran-aligned hackt...",Address,2025-07-02 12:05:33+00:00,2025-08-11T11:31:07Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,103.125.173.75,2025-08-11,DHA,1,DHA
9,6755399465229342,DHA Splunk API,2025-08-11T15:44:17Z,198.235.24.111,1639413,TASK0902923,Address,2025-07-28 17:16:05+00:00,2025-08-11T10:04:12Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,198.235.24.111,2025-08-11,DHA,7,"CDC, CMS, DHA, FDA, IHS, OS, VA"
16,5629499536034761,DHA Splunk API,2025-08-11T15:44:17Z,139.162.181.76,5,(U//FOUO) Federal law enforcement and the Inte...,Address,2025-04-11 17:47:40+00:00,2025-08-11T08:43:36Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,139.162.181.76,2025-08-11,DHA,1,DHA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3741,5263212,NIH Splunk API,2025-08-11T12:51:39Z,hcmiu.edu.vn,283,CSO received an email from CSIRC regarding a p...,Host,2025-01-22 18:05:29+00:00,2025-05-30T16:33:10Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NIH,hcmiu.edu.vn,2025-08-11,NIH,1,NIH
3742,5263228,NIH Splunk API,2025-08-11T12:51:39Z,metalhoz.com,288,CSO received an email from CSIRC regarding a p...,Host,2025-01-22 18:05:29+00:00,2025-05-30T16:33:09Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NIH,metalhoz.com,2025-08-11,NIH,1,NIH
3743,5629499542023970,DHA Splunk API,2025-08-11T15:44:17Z,chaturbate.com,7722,NaN,Stripped URL,2025-05-24 02:57:09+00:00,2025-05-27T08:35:54Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,chaturbate.com,2025-08-11,DHA,1,DHA
3744,5629499542023958,DHA Splunk API,2025-08-11T15:44:17Z,fmovies.co,199,NaN,Stripped URL,2025-05-24 02:50:36+00:00,2025-05-27T05:28:51Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,fmovies.co,2025-08-11,DHA,1,DHA


In [42]:
import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['indicator'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
filtered_recent_tags


No attributes found for indicator: 43.225.189.132
No attributes found for indicator: 45.134.140.131
No attributes found for indicator: 37.19.200.142
No attributes found for indicator: 185.187.243.217
No attributes found for indicator: 46.19.136.227
No attributes found for indicator: 37.19.200.151
No attributes found for indicator: minaffet.gov.rw
No attributes found for indicator: 169.150.196.16
No attributes found for indicator: yourpensionmeeting.com
No attributes found for indicator: 216.131.84.178
No attributes found for indicator: 62.182.99.151
No attributes found for indicator: 94.140.8.62
No attributes found for indicator: 45.144.113.239
No attributes found for indicator: 45.144.113.238
No attributes found for indicator: 5.181.234.220
No attributes found for indicator: 94.140.8.64
No attributes found for indicator: 94.140.8.61
No attributes found for indicator: 198.44.129.67
No attributes found for indicator: 45.152.180.244
No attributes found for indicator: 179.43.189.67
No att

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'


No attributes found for indicator: 172.98.33.195
No attributes found for indicator: 51.89.138.221
No attributes found for indicator: 146.70.45.166


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,tactics.data,tactics.count,platforms.data,platforms.count,OpDiv,indicator,obs_date,partner,partner_count,partners
0,6755399442005037,DHA Splunk API,2025-08-11T15:44:17Z,47.237.115.193,8695787,Suspicious attempts being made to authenticate...,Address,2025-02-12 20:18:21+00:00,2025-08-11T12:51:39Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,47.237.115.193,2025-08-11,DHA,7,"CMS, DHA, FDA, HHS, IHS, OS, VA"
1,5629499560349693,VA Splunk API,2025-08-10T23:06:33Z,103.172.81.132,224,"On July 18, security researchers and CISA disc...",Address,2025-07-22 17:27:41+00:00,2025-08-11T12:12:56Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,VA,103.172.81.132,2025-08-11,VA,1,VA
2,6755399460007958,DHA Splunk API,2025-08-11T15:44:17Z,103.125.173.75,5,"Details\nIn late June 2025, Iran-aligned hackt...",Address,2025-07-02 12:05:33+00:00,2025-08-11T11:31:07Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,103.125.173.75,2025-08-11,DHA,1,DHA
3,5629499536034761,DHA Splunk API,2025-08-11T15:44:17Z,139.162.181.76,5,(U//FOUO) Federal law enforcement and the Inte...,Address,2025-04-11 17:47:40+00:00,2025-08-11T08:43:36Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,139.162.181.76,2025-08-11,DHA,1,DHA
4,5629499536034798,DHA Splunk API,2025-08-11T15:44:17Z,172.104.251.198,6,(U//FOUO) Federal law enforcement and the Inte...,Address,2025-04-11 17:47:41+00:00,2025-08-11T08:43:34Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,172.104.251.198,2025-08-11,DHA,1,DHA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,5629499537015458,DHA Splunk API,2025-08-11T15:44:17Z,169.150.203.29,5892,NaN,Address,2025-04-23 15:01:06+00:00,2025-08-06T07:45:11Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,169.150.203.29,2025-08-11,DHA,1,DHA
184,4755432,OS Splunk API,2025-08-11T04:36:07Z,152.32.128.214,4213894,NaN,Address,2024-07-08 15:36:44+00:00,2025-08-06T07:45:09Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,OS,152.32.128.214,2025-08-11,OS,3,"CMS, HHS, OS"
185,4403793,VA Splunk API,2025-08-10T23:06:33Z,172.98.33.195,17518,NaN,Address,2023-09-19 10:33:49+00:00,2025-08-05T07:41:47Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,VA,172.98.33.195,2025-08-11,VA,1,VA
186,4403791,VA Splunk API,2025-08-10T23:06:33Z,51.89.138.221,745,NaN,Address,2023-09-19 10:33:49+00:00,2025-08-05T07:41:45Z,2025-08-11 00:00:00+00:00,...,NaN,NaN,NaN,NaN,VA,51.89.138.221,2025-08-11,VA,1,VA


In [43]:
# Enrich only final filtered indicators
indicator_values = filtered_recent_tags['summary'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_values)} indicators with DomainTools and VirusTotalV3...")

for value in indicator_values:
    try:
        # Use the indicator *value*, not the ID
        encoded_value = urllib.parse.quote(value)
        enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)

        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['summary'] = value  # Save the value as key
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched = pd.json_normalize(enriched_results)
    filtered_recent_tags = filtered_recent_tags.merge(df_enriched, on='summary', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in filtered_recent_tags.columns:
    enrichment_df = pd.json_normalize(
        filtered_recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = filtered_recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in filtered_recent_tags.columns]
    # Join enrichment columns back to filtered_recent_tags
    filtered_recent_tags = filtered_recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    filtered_recent_tags = filtered_recent_tags[filtered_recent_tags['vtMaliciousCount'] > 10]

    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 188 indicators with DomainTools and VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host link.mail.beehiiv.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host cdcfoundation.org cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host www.sthda.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host uploads-ssl.webflow.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host brppharma.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API 

Successfully enriched and merged 172 indicators.


In [44]:
import numpy as np

# Unnest the 'data.enrichment.data' column into separate columns for each enrichment type

def extract_enrichment(row):
    """Extracts enrichment fields from the list of dicts in 'data.enrichment.data'."""
    enrichments = row.get('data.enrichment.data')
    result = {}
    if isinstance(enrichments, list):
        for enrich in enrichments:
            enrich_type = enrich.get('type')
            if enrich_type == 'Shodan':
                # Flatten Shodan fields
                for key in ['hostNames', 'domains', 'tags', 'country', 'city', 'isp', 'asn', 'org', 'openPorts']:
                    result[f'shodan_{key}'] = enrich.get(key, np.nan)
            elif enrich_type == 'VirusTotal':
                result['vtMaliciousCount'] = enrich.get('vtMaliciousCount', np.nan)
    return pd.Series(result)

# Apply extraction to filtered_recent_tags
enrichment_expanded = filtered_recent_tags.apply(extract_enrichment, axis=1)
filtered_recent_tags = pd.concat([filtered_recent_tags, enrichment_expanded], axis=1)

filtered_recent_tags = filtered_recent_tags.rename(columns={
    'indicator': 'Indicator',
    'vtMaliciousCount': 'Malicious Score/Count',
    'obs_date': 'Observation Date',
    'shodan_asn': 'ASN',
    'rating': 'ThreatAssessRating',
    'confidence': 'ThreatAssessConfidence',
    'shodan_city': 'City',
    'shodan_country': 'Country',
    'data.legacyLink': 'ThreatConnect Link',
    'partners': 'Partners'
})

# Now select only the columns you want, after renaming
filtered_recent_tags = filtered_recent_tags[
    [
        'Indicator',
        'Malicious Score/Count',
        'Observation Date',
        'ASN',
        'ThreatAssessRating',
        'ThreatAssessConfidence',
        'City',
        'Country',
        'ThreatConnect Link',
        'Partners'
    ]
]

# Remove duplicate columns by keeping only the first occurrence
filtered_recent_tags = filtered_recent_tags.loc[:, ~filtered_recent_tags.columns.duplicated()]
filtered_recent_tags = filtered_recent_tags.fillna('unknown')
filtered_recent_tags = filtered_recent_tags.drop_duplicates()
filtered_recent_tags

,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
7,83.222.191.6,15.0,2025-08-11,AS204428,4.0,59.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, IHS, OS"
35,23.129.64.138,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
46,23.129.64.139,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
50,23.129.64.135,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
58,23.129.64.148,11.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
59,23.129.64.149,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
62,23.129.64.142,11.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
66,23.129.64.151,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,"FDA, OS, VA"
67,23.129.64.131,13.0,2025-08-11,AS396507,4.0,68.0,San Jose,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
73,185.220.101.185,11.0,2025-08-11,AS60729,4.0,62.0,Berlin,Germany,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA"


In [45]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in filtered_recent_tags['Partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: filtered_recent_tags[filtered_recent_tags['Partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: IHS (3 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
7,83.222.191.6,15.0,2025-08-11,AS204428,4.0,59.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, IHS, OS"
117,118.193.59.10,12.0,2025-08-11,AS135377,3.0,65.0,Frankfurt am Main,Germany,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HHS, IHS, OS"
133,83.222.191.2,13.0,2025-08-11,AS204428,4.0,60.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: DHA (2 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
7,83.222.191.6,15.0,2025-08-11,AS204428,4.0,59.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, IHS, OS"
133,83.222.191.2,13.0,2025-08-11,AS204428,4.0,60.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: FDA (5 records)


...

,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
7,83.222.191.6,15.0,2025-08-11,AS204428,4.0,59.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, IHS, OS"
66,23.129.64.151,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,"FDA, OS, VA"
82,178.20.55.16,11.0,2025-08-11,AS29075,5.0,96.0,Tremblay-en-France,France,https://hvs.threatconnect.com/auth/indicators/...,"CMS, OS"
93,80.67.167.81,12.0,2025-08-11,AS2027,4.0,63.0,Paris,France,https://hvs.threatconnect.com/auth/indicators/...,OS
104,45.84.107.128,11.0,2025-08-11,AS214503,4.0,60.0,Uppsala,Sweden,https://hvs.threatconnect.com/auth/indicators/...,"CMS, OS"
114,103.186.30.186,13.0,2025-08-11,AS141892,5.0,88.0,Bandung,Indonesia,https://hvs.threatconnect.com/auth/indicators/...,"FDA, OS"
115,45.141.215.40,11.0,2025-08-11,AS210558,3.0,73.0,Warsaw,Poland,https://hvs.threatconnect.com/auth/indicators/...,OS
117,118.193.59.10,12.0,2025-08-11,AS135377,3.0,65.0,Frankfurt am Main,Germany,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HHS, IHS, OS"
120,192.42.116.210,13.0,2025-08-11,AS1101,3.0,66.0,Nieuwegein,Netherlands,https://hvs.threatconnect.com/auth/indicators/...,"CMS, OS"
133,83.222.191.2,13.0,2025-08-11,AS204428,4.0,60.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: VA (9 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
35,23.129.64.138,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
46,23.129.64.139,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
50,23.129.64.135,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
58,23.129.64.148,11.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
59,23.129.64.149,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
62,23.129.64.142,11.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
66,23.129.64.151,12.0,2025-08-11,AS396507,4.0,68.0,Fremont,United States,https://hvs.threatconnect.com/auth/indicators/...,"FDA, OS, VA"
67,23.129.64.131,13.0,2025-08-11,AS396507,4.0,68.0,San Jose,United States,https://hvs.threatconnect.com/auth/indicators/...,VA
133,83.222.191.2,13.0,2025-08-11,AS204428,4.0,60.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: HHS (3 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
117,118.193.59.10,12.0,2025-08-11,AS135377,3.0,65.0,Frankfurt am Main,Germany,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, HHS, IHS, OS"
133,83.222.191.2,13.0,2025-08-11,AS204428,4.0,60.0,Sofia,Bulgaria,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
151,194.180.48.211,14.0,2025-08-11,unknown,3.0,0.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, HHS"


In [46]:
import os
import re
from datetime import datetime
import pandas as pd

Tippers_Path = r'C:\Users\jaskew\Documents\TipperTemp'
today_str = datetime.utcnow().strftime("%Y%m%d")
date_folder = os.path.join(Tippers_Path, today_str)
os.makedirs(date_folder, exist_ok=True)

for partner, df in partner_buckets.items():
    safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]
    partner_folder = os.path.join(date_folder, safe_partner)
    os.makedirs(partner_folder, exist_ok=True)
    excel_path = os.path.join(partner_folder, f"{safe_partner}.xlsx")

    # Remove timezone for Excel compatibility
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name=safe_partner, index=False)
        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)
        for i, col in enumerate(df.columns):
            if not df.empty:
                col_lens = df[col].fillna("").astype(str).apply(len)
                max_len = max(col_lens.max(), len(str(col)))
            else:
                max_len = len(str(col))
            worksheet.set_column(i, i, min(max_len + 2, 50))

    print(f"Saved: {excel_path}")


Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\IHS\IHS.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\DHA\DHA.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\FDA\FDA.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\CMS\CMS.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\OS\OS.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\VA\VA.xlsx
Saved: C:\Users\jaskew\Documents\TipperTemp\20250811\HHS\HHS.xlsx
